In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# -*- coding: utf-8 -*-
# ============================================
# Full-SNN TTFS NeuMF (Amp-Preserving, 2-Stage Cascade) — ACC↑ + ENERGY↓↓
# - S1: small T + light gating -> Dynamic Top-M (per-user, margin-based)
# - S2: larger T on Top-M (full open gating)
# - Full-stack TTFS: Embedding -> MLP (1st dual, hidden single) -> GMF dual -> Readout
# - Stage-specific gamma (γ) + Adaptive-α (per user) in S2
# - vt-only re-rank for top-L (cheap tie-resolution)
# - Top-L ANN tail re-rank (very small MAC; energy accounted)
# - AMP (torch.amp.autocast) + big batch
# - Fixed negative pool (ANN/SNN 동일 후보), /user 공정 회계
# - Flexible checkpoint loader (embed slice/pad + sanity-fix
# - 유효 이벤트만 SynOp/메모리/에너지로 집계 + 조기 기각(upper bound) 반영
# ============================================

import os, time, math, random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch import amp
from typing import Dict, List, Tuple
from scipy import sparse
from scipy.sparse import coo_matrix
from tqdm import tqdm
import warnings

# ---------------- Paths & dataset ----------------
DATA_ROOT  = "/content/drive/MyDrive/NCF/LightGCN-PyTorch/data"
DATASET    = "gowalla"
CKPT_IN    = "/content/drive/MyDrive/NCF/lightgcn_test.pth"

# ---------------- Eval params ----------------
USERS_SAMPLE   = 500
NUM_NEGATIVES  = 1000
K              = 10
BATCH_SIZE     = 1536

# ---------------- TTFS / dynamics ----------------
TTFS_PERCENTILE = 99.2
MAG_EPS         = 3e-4
NEG_SHIFT_FRAC  = 0.0

ALPHA_HYBRID_INIT = 0.56

GAMMA_MLP_IN_S1 = 1.18
GAMMA_MLP_IN_S2 = 1.34
GAMMA_GMF_S1    = 1.12
GAMMA_GMF_S2    = 1.24

# ---- 2-Stage SNN cascade ----
T1              = 160    # Stage-1 tick
T2              = 704    # Stage-2 tick

# 동적 Top-M 범위 (per-user)
TOP_M_LO        = 128
TOP_M_BASE      = 320
TOP_M_HI        = 512
MARGIN_PROBE    = 96
MARGIN_HI       = 0.20
MARGIN_LO       = 0.08

S1_KEEP_FIRST   = 0.70
S1_KEEP_HID     = 0.65
S1_KEEP_GMF     = 1.00
S1_KEEP_MLP     = 0.70

S2_KEEP_FIRST   = 1.00
S2_KEEP_HID     = 1.00
S2_KEEP_GMF     = 1.00
S2_KEEP_MLP     = 1.00

DITHER_K        = 1

# ---- vt-only 재정렬  ----
TOP_L_VT_RERANK = 192

# ---- Adaptive alpha (S2) ----
ADAPTIVE_ALPHA       = True
ALPHA_MIN, ALPHA_MAX = 0.44, 0.62
ADAPTIVE_ALPHA_TOPK  = 96


# ---- ANN 꼬리 재정렬 ----
ENABLE_ANN_TAIL    = False
TAIL_L             = 32

# ---- Energy / chip params ----
ANN_REUSE_FACTOR   = 25
E_MAC              = 1.0
E_AC               = 0.06
E_LOCAL_MEM_RW     = 0.2
E_DISTANT_MEM_RW   = 1.0

SEED = 2025
warnings.filterwarnings("ignore", category=UserWarning)

# ========== Utils ==========
def std_err(x):
    x = np.array(x);  return np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0


# ========== loaders ==========
def parse_lines(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            s=raw.strip()
            if not s: continue
            parts=s.split()
            uid=int(parts[0])
            for tok in parts[1:]:
                if tok: yield uid, int(tok)

def load_as_coo(data_root, dataset):
    # 불러오기
    ddir=os.path.join(data_root, dataset)
    tr_path=os.path.join(ddir,"train.txt"); te_path=os.path.join(ddir,"test.txt")
    if not (os.path.exists(tr_path) and os.path.exists(te_path)):
        raise FileNotFoundError(f"Expected train.txt/test.txt under {ddir}")


    tr_pairs=list(parse_lines(tr_path))
    te_pairs=list(parse_lines(te_path))
    users=sorted({u for u,_ in tr_pairs} | {u for u,_ in te_pairs})
    items=sorted({i for _,i in tr_pairs} | {i for _,i in te_pairs})

    u_map={u:i for i,u in enumerate(users)}
    v_map={v:i for i,v in enumerate(items)}
    n_users, n_items = len(u_map), len(v_map)

    tr_u=np.fromiter((u_map[u] for u,_ in tr_pairs), dtype=np.int64, count=len(tr_pairs))
    tr_v=np.fromiter((v_map[v] for _,v in tr_pairs), dtype=np.int64, count=len(tr_pairs))
    te_u=np.fromiter((u_map[u] for u,_ in te_pairs), dtype=np.int64, count=len(te_pairs))
    te_v=np.fromiter((v_map[v] for _,v in te_pairs), dtype=np.int64, count=len(te_pairs))

    Rf_train=sparse.coo_matrix((np.ones_like(tr_u, dtype=np.float32),(tr_u,tr_v)), shape=(n_users,n_items), dtype=np.float32)
    Rf_test =sparse.coo_matrix((np.ones_like(te_u, dtype=np.float32),(te_u,te_v)), shape=(n_users,n_items), dtype=np.float32)


    return Rf_train, Rf_test, n_users, n_items


def build_trainset_testmap_from_coo(Rf_train, Rf_test):

    train_set=set(zip(Rf_train.row.tolist(), Rf_train.col.tolist()))
    test_df=pd.DataFrame({'user':Rf_test.row, 'item':Rf_test.col})
    test_map=test_df.groupby('user')['item'].apply(list).to_dict()

    return train_set, test_map

# ========== NeuMF  ==========

class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, gmf_dim=8, mlp_dim=32, mlp_layers=[64,32,16,8]):
        super().__init__()
        self.gmf_user_emb = nn.Embedding(num_users, gmf_dim)
        self.gmf_item_emb = nn.Embedding(num_items, gmf_dim)
        self.mlp_user_emb = nn.Embedding(num_users, mlp_dim)
        self.mlp_item_emb = nn.Embedding(num_items, mlp_dim)
        self.mlp_layer_dims=[]
        input_size=mlp_dim*2; layers=[]
        for h in mlp_layers:
            self.mlp_layer_dims.append((input_size,h))
            layers += [nn.Linear(input_size,h), nn.ReLU()]
            input_size=h
        self.mlp_layers = nn.Sequential(*layers)
        self.gmf_dim=gmf_dim; self.mlp_dim=mlp_dim
        self.mlp_last_dim = mlp_layers[-1] if mlp_layers else mlp_dim*2
        self.out = nn.Linear(self.gmf_dim + self.mlp_last_dim, 1)

    def get_mlp_input(self, u, v):
        return torch.cat([self.mlp_user_emb(u), self.mlp_item_emb(v)], dim=-1)

    def get_gmf_only(self, u, v):
        return self.gmf_user_emb(u) * self.gmf_item_emb(v)

    def get_features(self, u, v):
        g=self.get_gmf_only(u,v); m=self.mlp_layers(self.get_mlp_input(u,v))
        return g,m

    def forward(self, u, v):
        g,m=self.get_features(u,v); x=torch.cat([g,m],dim=-1)
        return torch.sigmoid(self.out(x))

# 가중치 가져오기
def _infer_neumf_arch_from_state_dict(sd):
    gmf_user_w = sd.get("gmf_user_emb.weight", sd.get("gmf_user_embedding.weight", None))
    gmf_item_w = sd.get("gmf_item_emb.weight", sd.get("gmf_item_embedding.weight", None))
    mlp_user_w = sd.get("mlp_user_emb.weight", sd.get("mlp_user_embedding.weight", None))
    mlp_item_w = sd.get("mlp_item_emb.weight", sd.get("mlp_item_embedding.weight", None))

    if any(x is None for x in [gmf_user_w, gmf_item_w, mlp_user_w, mlp_item_w]):
        raise RuntimeError("Checkpoint missing embeddings.")


    gmf_dim=int(gmf_user_w.shape[1]); mlp_dim=int(mlp_user_w.shape[1])
    mlp_blocks=[]

    for k,v in sd.items():
        if k.startswith("mlp_layers.") and k.endswith(".weight") and getattr(v,"ndim",0)==2:
            try: idx=int(k.split(".")[1])
            except: continue
            mlp_blocks.append((idx, int(v.shape[0]), int(v.shape[1])))

    if mlp_blocks:
        mlp_blocks.sort(key=lambda x:x[0]); mlp_layers=[out for _,out,_ in mlp_blocks]

    else:
        out_w = sd.get("out.weight", sd.get("output_layer.weight", None))
        if out_w is None: raise RuntimeError("No out.weight in ckpt")
        mlp_last_dim = int(out_w.shape[1]) - gmf_dim
        if mlp_last_dim<=0: raise RuntimeError("infer mlp_last_dim failed")
        mlp_layers=[mlp_last_dim]

    return gmf_dim, mlp_dim, mlp_layers

def _remap_ckpt_keys(sd):
    new_sd={}; head=None
    if "out_click.weight" in sd: head="out_click"
    elif "out_buy.weight"  in sd: head="out_buy"
    elif "output_layer.weight" in sd: head="output_layer"
    elif "out.weight" in sd: head="out"
    for k,v in sd.items():
        nk=(k.replace("gmf_user_embedding","gmf_user_emb")
              .replace("gmf_item_embedding","gmf_item_emb")
              .replace("mlp_user_embedding","mlp_user_emb")
              .replace("mlp_item_embedding","mlp_item_emb")
              .replace("output_layer","out"))
        if head in ("out_click","out_buy"):
            if k.startswith(head+"."): nk="out."+k.split(".",1)[1]
            elif k.startswith("out.") or k.startswith("output_layer."): continue
        new_sd[nk]=v
    return new_sd

def _extract_mlp_linear_from_sd(sd):
    pairs={}
    for k,v in sd.items():
        if k.startswith("mlp_layers.") and (k.endswith(".weight") or k.endswith(".bias")):
            try: idx=int(k.split(".")[1])
            except: continue
            d=pairs.setdefault(idx,{})
            if k.endswith(".weight"): d["W"]=v
            else: d["b"]=v
    blocks=[]
    for idx in sorted(pairs.keys()):
        d=pairs[idx]
        if "W" in d and "b" in d and getattr(d["W"],"ndim",0)==2:
            blocks.append((idx,d["W"],d["b"]))
    return blocks

def _copy_linear_safely(lin: nn.Linear, Wsrc, bsrc):
    with torch.no_grad():
        h,w=lin.weight.shape; H,W=Wsrc.shape
        lin.weight.zero_(); lin.bias.zero_()
        lin.weight[:min(h,H), :min(w,W)].copy_(Wsrc[:min(h,H), :min(w,W)])
        lin.bias[:min(lin.bias.shape[0], bsrc.shape[0])].copy_(bsrc[:min(lin.bias.shape[0], bsrc.shape[0])])

def _embed_sanity_fix(model: "NeuMF", n_users, n_items):
    def _ensure_embed(module: nn.Embedding, want_n: int, dim: int, name: str):
        if module.num_embeddings == want_n and module.embedding_dim == dim:
            return module
        new = nn.Embedding(want_n, dim)
        with torch.no_grad():
            new.weight.zero_()
            copy_n = min(want_n, module.weight.shape[0])
            new.weight[:copy_n] = module.weight[:copy_n]
            if want_n > module.weight.shape[0]:
                nn.init.normal_(new.weight[copy_n:], mean=0.0, std=0.01)
        print(f"[sanity] reinit {name}: {module.num_embeddings}->{want_n}")
        return new.to(module.weight.device)
    model.gmf_user_emb = _ensure_embed(model.gmf_user_emb, n_users, model.gmf_dim, "gmf_user_emb")
    model.gmf_item_emb = _ensure_embed(model.gmf_item_emb, n_items, model.gmf_dim, "gmf_item_emb")
    model.mlp_user_emb = _ensure_embed(model.mlp_user_emb, n_users, model.mlp_dim, "mlp_user_emb")
    model.mlp_item_emb = _ensure_embed(model.mlp_item_emb, n_items, model.mlp_dim, "mlp_item_emb")
    return model

# 복원 로더
def load_neumf_checkpoint_safely(model_path, n_users, n_items, device="cpu"):
    state=torch.load(model_path, map_location="cpu")
    sd=state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state
    sd=_remap_ckpt_keys(sd)
    gmf_dim, mlp_dim, mlp_layers=_infer_neumf_arch_from_state_dict(sd)
    model=NeuMF(n_users, n_items, gmf_dim=gmf_dim, mlp_dim=mlp_dim, mlp_layers=mlp_layers).to("cpu")

    msd=model.state_dict()
    def _copy_embed(param_name, ckpt_name):
        if ckpt_name not in sd or param_name not in msd: return
        W_src=sd[ckpt_name]
        W_tgt=msd[param_name]
        N_src,D_src=W_src.shape
        N_tgt,D_tgt=W_tgt.shape
        assert D_src==D_tgt, f"embed dim mismatch: {param_name}"

        if N_src>=N_tgt:
            W_new=W_src[:N_tgt].clone(); info=f"slice {ckpt_name} {N_src}->{N_tgt}"

        else:
            W_new=W_tgt.clone(); W_new.zero_(); W_new[:N_src]=W_src
            nn.init.normal_(W_new[N_src:], mean=0.0, std=0.01); info=f"pad {ckpt_name} {N_src}->{N_tgt}"
        msd[param_name]=W_new; print(f"[embed] {info}")

    _copy_embed("gmf_user_emb.weight","gmf_user_emb.weight")
    _copy_embed("gmf_item_emb.weight","gmf_item_emb.weight")
    _copy_embed("mlp_user_emb.weight","mlp_user_emb.weight")
    _copy_embed("mlp_item_emb.weight","mlp_item_emb.weight")

    blocks=_extract_mlp_linear_from_sd(sd)
    linears=[m for m in model.mlp_layers if isinstance(m, nn.Linear)]

    for j in range(min(len(blocks), len(linears))):
        _,Wsrc,bsrc=blocks[j]; _copy_linear_safely(linears[j], Wsrc, bsrc)

    if "out.weight" in sd and "out.bias" in sd:
        _copy_linear_safely(model.out, sd["out.weight"], sd["out.bias"])

    model.load_state_dict(msd, strict=False)
    model=_embed_sanity_fix(model, n_users, n_items).to(device)
    print(f"[CKPT] flexible-load + sanity-fix: users={n_users}, items={n_items}, gmf={gmf_dim}, mlp={mlp_dim}, mlp_layers={mlp_layers}")
    return model

# Calibration
@torch.no_grad()
def fit_scale_vec(ncf, train_set, device, p=TTFS_PERCENTILE, what="mlp_in",
                  max_users=300, items_per_user=256, seed=SEED):
    rng = np.random.default_rng(seed)
    user_set = sorted({u for (u,_) in train_set})
    users = rng.choice(user_set, size=min(max_users, len(user_set)), replace=False).tolist()
    BUF=[]
    for u in users:
        items = [i for (uu,i) in train_set if uu==u]
        if not items: continue
        if len(items)>items_per_user:
            items = rng.choice(items, size=items_per_user, replace=False).tolist()
        U=torch.tensor([u]*len(items), device=device, dtype=torch.long)
        I=torch.tensor(items,       device=device, dtype=torch.long)
        if what=="mlp_in":
            X = ncf.get_mlp_input(U,I)
        elif what=="gmf":
            X = ncf.get_gmf_only(U,I)
        else:
            raise ValueError
        BUF.append(X.detach().float().cpu())
    ALL = torch.cat(BUF,0) if BUF else torch.zeros((1, ncf.mlp_dim*2 if what=="mlp_in" else ncf.gmf_dim))
    s = np.percentile(np.abs(ALL.numpy()), p, axis=0).astype(np.float32); s[s<1e-8]=1e-8
    return torch.from_numpy(s).to(device)

# ========== TTFS helpers ==========
@torch.no_grad()
def ttfs_times_posneg(features, T, scale_vec, gamma, eps=MAG_EPS):
    x=features; s=scale_vec.view(1,-1).to(x.device, x.dtype)
    pos = torch.clamp(x,  min=0.0)
    neg = torch.clamp(-x, min=0.0)
    mag_p = torch.clamp(pos/torch.clamp(s,1e-8), max=1.0)
    mag_n = torch.clamp(neg/torch.clamp(s,1e-8), max=1.0)
    if eps>0.0:
        mag_p = torch.where(mag_p<eps, torch.zeros_like(mag_p), mag_p)
        mag_n = torch.where(mag_n<eps, torch.zeros_like(mag_n), mag_n)
    if gamma!=1.0:
        mag_p = torch.pow(mag_p, gamma); mag_n = torch.pow(mag_n, gamma)
    t_p = torch.round((1.0 - mag_p) * (T-1)).long().clamp(0, T-1)
    t_n = torch.round((1.0 - mag_n) * (T-1)).long().clamp(0, T-1)
    mask_p = (mag_p > 0); mask_n = (mag_n > 0)
    return t_p, mask_p, t_n, mask_n

@torch.no_grad()
def _time_to_amp(t: torch.Tensor, T: int):
    amp = 1.0 - (t.float() / max(1.0, (T-1.0)))
    return torch.clamp(amp, min=0.0, max=1.0) * (t < T).float()

# 임계값 기준 몇%만 계산할건지
def _apply_topk_gate_imp(imp: torch.Tensor, keep_frac: float):
    if keep_frac >= 1.0: return torch.ones_like(imp, dtype=torch.bool)
    if keep_frac <= 0.0: return torch.zeros_like(imp, dtype=torch.bool)
    B,D = imp.shape
    k = max(1, int(math.ceil(keep_frac * D)))
    thresh = torch.topk(imp, k=k, dim=1).values[:, -1].unsqueeze(1)
    return (imp >= thresh)

# ========== early 이벤트 집계 ==========
@torch.no_grad()
def _used_events_with_early_reject(c_sorted: torch.Tensor, csum: torch.Tensor, b_or_thr, first_idx, any_hit, per_output=True):
    if per_output:
        B, L, D = c_sorted.shape
        pos_part = torch.clamp(c_sorted, min=0.0)
        pos_future_incl = torch.flip(torch.cumsum(torch.flip(pos_part, dims=[1]), dim=1), dims=[1])
        pos_after = torch.zeros_like(pos_future_incl); pos_after[:, :-1, :] = pos_future_incl[:, 1:, :]
        thr = (-b_or_thr.view(1,1,D)).expand_as(csum)
        still_possible = (csum + pos_after) >= thr
        rej = ~still_possible
        any_rej = rej.any(dim=1)
        first_rej = rej.float().argmax(dim=1)

        eff_mask = (c_sorted.abs() > 0)
        arangeL  = torch.arange(L, device=c_sorted.device).view(1,L,1)
        le_hit   = arangeL <= first_idx.unsqueeze(1)
        le_rej   = arangeL <= first_rej.unsqueeze(1)
        eff_until_hit = (eff_mask & le_hit).sum(dim=1)
        eff_until_rej = (eff_mask & le_rej).sum(dim=1)
        eff_all       =  eff_mask.sum(dim=1)
        used_eff = torch.where(any_hit, eff_until_hit,
                               torch.where(any_rej, eff_until_rej, eff_all))
        return used_eff.int()

    else:
        B, L = c_sorted.shape[0], c_sorted.shape[1]
        pos_part = torch.clamp(c_sorted, min=0.0)
        pos_future_incl = torch.flip(torch.cumsum(torch.flip(pos_part, dims=[1]), dim=1), dims=[1])
        pos_after = torch.zeros_like(pos_future_incl); pos_after[:, :-1] = pos_future_incl[:, 1:]
        thr = float(-b_or_thr.view(-1)[0])
        still_possible = (csum + pos_after) >= thr
        rej = ~still_possible
        any_rej = rej.any(dim=1)
        first_rej = rej.float().argmax(dim=1)

        eff_mask = (c_sorted.abs() > 0)
        arangeL  = torch.arange(L, device=c_sorted.device).view(1,L)
        le_hit   = arangeL <= first_idx.unsqueeze(1)
        le_rej   = arangeL <= first_rej.unsqueeze(1)
        eff_until_hit = (eff_mask & le_hit).sum(dim=1)
        eff_until_rej = (eff_mask & le_rej).sum(dim=1)
        eff_all       =  eff_mask.sum(dim=1)
        used_eff = torch.where(any_hit, eff_until_hit,
                               torch.where(any_rej, eff_until_rej, eff_all))
        return used_eff.int()

# ========== Layer kernels  ==========
@torch.no_grad()
def layer_first_signed_inputs_amp(t_pos, m_pos, t_neg, m_neg, W, b, T,
                                  neg_shift_frac=NEG_SHIFT_FRAC,
                                  keep_frac=1.0,
                                  dither_k=1):
    device=t_pos.device; B,Din=t_pos.shape; Dout=W.shape[1]
    neg_shift = int(max(0, math.floor(neg_shift_frac * T)))

    amp_pos = _time_to_amp(t_pos, T) * m_pos.float()
    amp_neg = _time_to_amp(t_neg, T) * m_neg.float()

    imp_pos = torch.abs(W).sum(dim=1).view(1,-1) * amp_pos
    imp_neg = torch.abs(W).sum(dim=1).view(1,-1) * amp_neg
    mask_pos = _apply_topk_gate_imp(imp_pos, keep_frac)
    mask_neg = _apply_topk_gate_imp(imp_neg, keep_frac)
    amp_pos = amp_pos * mask_pos.float()
    amp_neg = amp_neg * mask_neg.float()

    tpos = t_pos.unsqueeze(2).expand(B,Din,Dout)
    tneg = t_neg.unsqueeze(2).expand(B,Din,Dout)
    if neg_shift>0:
        eff_pos = (W > 0).unsqueeze(0).expand(B,Din,Dout)
        eff_neg = (-W > 0).unsqueeze(0).expand(B,Din,Dout)
        tpos = torch.where(eff_pos, tpos, torch.clamp(tpos + neg_shift, max=T-1))
        tneg = torch.where(eff_neg, tneg, torch.clamp(tneg + neg_shift, max=T-1))

    c_pos = W.unsqueeze(0) * amp_pos.unsqueeze(2)
    c_neg = (-W).unsqueeze(0) * amp_neg.unsqueeze(2)

    if dither_k and dither_k>1:
        tpos = tpos.unsqueeze(3).expand(B,Din,Dout,dither_k).reshape(B, Din*dither_k, Dout)
        tneg = tneg.unsqueeze(3).expand(B,Din,Dout,dither_k).reshape(B, Din*dither_k, Dout)
        c_pos= c_pos.unsqueeze(3).expand(B,Din,Dout,dither_k).reshape(B, Din*dither_k, Dout) / float(dither_k)
        c_neg= c_neg.unsqueeze(3).expand(B,Din,Dout,dither_k).reshape(B, Din*dither_k, Dout) / float(dither_k)

    times   = torch.cat([tpos, tneg], dim=1)
    contrib = torch.cat([c_pos, c_neg], dim=1)
    L_total = times.shape[1]

    sort_t, idx = torch.sort(times, dim=1, stable=True)
    c_sorted = torch.gather(contrib, 1, idx)
    csum     = torch.cumsum(c_sorted, dim=1)
    vT_pre   = csum[:, -1, :] + b.view(1,1,Dout)

    thr = (-b.view(1,1,Dout)).expand_as(csum)
    hit = (csum >= thr); any_hit = hit.any(dim=1)
    first_idx = hit.float().argmax(dim=1)

    t_sel = sort_t.gather(1, first_idx.unsqueeze(1)).squeeze(1).float()
    t_out = torch.full((B,Dout), float(T), device=device)
    t_out[any_hit] = t_sel[any_hit]

    used_eff = _used_events_with_early_reject(c_sorted, csum, b, first_idx, any_hit, per_output=True)
    return t_out, used_eff, vT_pre, L_total

@torch.no_grad()
def layer_relu_times_amp(t_pre, W, b, T,
                         neg_shift_frac=NEG_SHIFT_FRAC,
                         keep_frac=1.0,
                         dither_k=1):
    device=t_pre.device; B,Din=t_pre.shape; Dout=W.shape[1]
    neg_shift = int(max(0, math.floor(neg_shift_frac * T)))

    amp = _time_to_amp(t_pre, T)
    imp = torch.abs(W).sum(dim=1).view(1,-1) * amp
    mask = _apply_topk_gate_imp(imp, keep_frac)
    amp  = amp * mask.float()

    t_eff = t_pre.unsqueeze(2).expand(B,Din,Dout)
    if neg_shift>0:
        eff = (W > 0).unsqueeze(0).expand(B,Din,Dout)
        t_eff = torch.where(eff, t_eff, torch.clamp(t_eff + neg_shift, max=T-1))

    contrib = W.unsqueeze(0) * amp.unsqueeze(2)

    if dither_k and dither_k>1:
        t_eff = t_eff.unsqueeze(3).expand(B,Din,Dout,dither_k).reshape(B, Din*dither_k, Dout)
        contrib = contrib.unsqueeze(3).expand(B,Din,Dout,dither_k).reshape(B, Din*dither_k, Dout) / float(dither_k)

    L_total = t_eff.shape[1]
    sort_t, idx = torch.sort(t_eff, dim=1, stable=True)
    c_sorted = torch.gather(contrib, 1, idx)
    csum     = torch.cumsum(c_sorted, dim=1)
    vT_pre   = csum[:, -1, :] + b.view(1,1,Dout)

    thr = (-b.view(1,1,Dout)).expand_as(csum)
    hit = (csum >= thr); any_hit  = hit.any(dim=1)
    first_idx= hit.float().argmax(dim=1)

    t_sel = sort_t.gather(1, first_idx.unsqueeze(1)).squeeze(1).float()
    t_out = torch.full((B,Dout), float(T), device=device)
    t_out[any_hit] = t_sel[any_hit]

    used_eff = _used_events_with_early_reject(c_sorted, csum, b, first_idx, any_hit, per_output=True)
    return t_out, used_eff, vT_pre, L_total


# 시간과 발화 전환
@torch.no_grad()
def readout_merge_signed_amp(tg_pos, mg_pos, tg_neg, mg_neg, tm_last, w_m, w_g, b_out, T,
                             keep_frac_gmf=1.0, keep_frac_mlp=1.0, dither_k=1):
    device = w_g.device
    B, Dg = tg_pos.shape
    Dm = tm_last.shape[1] if tm_last.numel()>0 else 0

    amp_gp = _time_to_amp(tg_pos, T) * mg_pos.float()
    amp_gn = _time_to_amp(tg_neg, T) * mg_neg.float()
    amp_m  = _time_to_amp(tm_last, T) if Dm>0 else None

    wg = w_g.abs().view(1,-1)
    mask_gp = _apply_topk_gate_imp(amp_gp*wg, keep_frac_gmf)
    mask_gn = _apply_topk_gate_imp(amp_gn*wg, keep_frac_gmf)
    amp_gp  = amp_gp * mask_gp.float()
    amp_gn  = amp_gn * mask_gn.float()

    if Dm>0:
        wm = w_m.abs().view(1,-1)
        mask_m = _apply_topk_gate_imp(amp_m*wm, keep_frac_mlp)
        amp_m  = amp_m * mask_m.float()

    tgp = tg_pos.unsqueeze(2).expand(B,Dg,1)
    tgn = tg_neg.unsqueeze(2).expand(B,Dg,1)
    c_gp= w_g.view(1,Dg,1) * amp_gp.unsqueeze(2)
    c_gn= (-w_g).view(1,Dg,1) * amp_gn.unsqueeze(2)

    if Dm>0:
        tm3 = tm_last.unsqueeze(2).expand(B,Dm,1)
        cm  = w_m.view(1,Dm,1) * amp_m.unsqueeze(2)
        times  = torch.cat([tgp, tgn, tm3], dim=1)
        contrib= torch.cat([c_gp, c_gn, cm],  dim=1)
    else:
        times  = torch.cat([tgp, tgn], dim=1)
        contrib= torch.cat([c_gp, c_gn], dim=1)

    if dither_k and dither_k>1:
        times   = times.expand(B, times.shape[1], dither_k).reshape(B, -1, 1)
        contrib = contrib.expand(B, contrib.shape[1], dither_k).reshape(B, -1, 1) / float(dither_k)

    L_total = times.shape[1]
    sort_t, idx = torch.sort(times, dim=1, stable=True)
    c_sorted = torch.gather(contrib, 1, idx).squeeze(2)
    csum     = torch.cumsum(c_sorted, dim=1)
    vT       = csum[:, -1] + b_out.view(-1)[0]

    thr = float(-b_out.view(-1)[0])
    hit = (csum >= thr); any_hit = hit.any(dim=1)
    first_idx = hit.float().argmax(dim=1)

    t_sel = sort_t.gather(1, first_idx.view(-1,1).unsqueeze(2)).squeeze(2).squeeze(1).float()
    t_out = torch.full((B,), float(T), device=device)
    t_out[any_hit] = t_sel[any_hit]

    used_eff = _used_events_with_early_reject(c_sorted, csum, b_out, first_idx, any_hit, per_output=False)
    return t_out, used_eff, vT, L_total

# ========== Candidate pool ==========
@torch.no_grad()
def build_fixed_eval_pool(test_map, train_set, n_items,
                          users_sample, num_neg, seed = SEED):
    rng = np.random.default_rng(seed)
    test_users = list(test_map.keys())
    if users_sample and users_sample < len(test_users):
        test_users = rng.choice(test_users, size=users_sample, replace=False).tolist()
    fixed_negs: Dict[int, List[int]] = {}
    for u in test_users:
        pos_items = test_map.get(u, [])
        if not pos_items:
            continue
        pos = pos_items[0]
        cand = []
        while len(cand) < num_neg:
            i = int(rng.integers(0, n_items))
            if i != pos and (u, i) not in train_set:
                cand.append(i)
        fixed_negs[u] = cand
    return test_users, fixed_negs

# ========== ANN ==========
@torch.no_grad()
def evaluate_ann_full(model: NeuMF, test_map, train_set, n_items, device):
    model.eval(); model = model.to(device)
    users, fixed_negs = build_fixed_eval_pool(test_map, train_set, n_items, USERS_SAMPLE, NUM_NEGATIVES, SEED)

    hits, ndcgs = [], []
    macs_total = 0
    mem_access = 0
    t0=time.time()

    for u in tqdm(users, desc="ANN eval", leave=False):
        pos_items = test_map.get(u, [])
        if not pos_items: continue
        pos = pos_items[0]
        eval_items = [pos] + fixed_negs[u]
        uu = torch.full((len(eval_items),), u, device=device, dtype=torch.long)
        vv = torch.tensor(eval_items, device=device, dtype=torch.long)

        g = model.get_gmf_only(uu, vv)
        x = model.get_mlp_input(uu, vv)
        for layer in model.mlp_layers:
            if isinstance(layer, nn.Linear):
                macs_total += layer.in_features * layer.out_features * x.shape[0]
            x = layer(x)
        out = model.out(torch.cat([g, x], dim=-1)).squeeze(-1)
        macs_total += (g.shape[1] + x.shape[1]) * 1 * x.shape[0]
        scores = out.detach().float().cpu().numpy()

        mem_access += (2 * model.gmf_user_emb.embedding_dim + 2 * model.mlp_user_emb.embedding_dim)
        order = np.argsort(-scores)
        rank = np.where(order == 0)[0][0]
        hit = 1.0 if rank < K else 0.0
        ndcg = 1.0 / math.log2(rank + 2) if rank < K else 0.0
        hits.append(hit); ndcgs.append(ndcg)

    denom = max(1,len(users))
    latency_ms_per_user = ((time.time()-t0)*1000.0)/denom
    energy_est = (E_MAC * (macs_total/denom) + E_LOCAL_MEM_RW * (mem_access/denom))
    return {
        "Hit@K": float(np.mean(hits)) if hits else 0.0,
        "NDCG@K": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "Latency(ms/user)": latency_ms_per_user,
        "Ops (MACs)": float(macs_total/denom),
        "Memory Access Count": float(mem_access/denom),
        "Estimated Energy Score (ANN full)": float(energy_est),
    }

# Top-M
@torch.no_grad()
def _pick_dynamic_topM(scores,
                       lo=TOP_M_LO, base=TOP_M_BASE, hi=TOP_M_HI,
                       probe=MARGIN_PROBE, margin_lo=MARGIN_LO, margin_hi=MARGIN_HI):
    s_sorted, _ = torch.sort(scores, descending=True)
    if s_sorted.numel() <= 1:
        return base
    j = min(probe-1, s_sorted.numel()-1)
    margin = (s_sorted[0] - s_sorted[j]).item()
    if margin >= margin_hi:
        return lo
    if margin <= margin_lo:
        return hi
    t = (margin_hi - margin) / max(1e-8, (margin_hi - margin_lo))
    M = int(round(lo + (hi - lo) * t))
    return int(max(lo, min(hi, M)))

# Adaptive alpha util
@torch.no_grad()
def _pearson_corr(x, y) -> torch.Tensor:
    x = x - x.mean()
    y = y - y.mean()
    denom = (x.norm() * y.norm()).clamp(min=1e-8)
    return (x @ y) / denom

@torch.no_grad()
def _adaptive_alpha_from_agreement(t_part, v_part,
                                   k=ADAPTIVE_ALPHA_TOPK,
                                   a_min=ALPHA_MIN, a_max=ALPHA_MAX) -> float:

    n = t_part.numel()
    if n <= 1:
        return float((a_min + a_max) * 0.5)
    L = min(k, n)

    vals_t, idx = torch.topk(t_part, k=L, largest=True)
    vals_v = v_part[idx]


    ord_t = torch.argsort(vals_t, descending=True)
    ord_v = torch.argsort(vals_v, descending=True)
    rank_t = torch.empty_like(ord_t, dtype=torch.float32); rank_t[ord_t] = torch.arange(L, device=vals_t.device, dtype=torch.float32)
    rank_v = torch.empty_like(ord_v, dtype=torch.float32); rank_v[ord_v] = torch.arange(L, device=vals_v.device, dtype=torch.float32)
    rho = _pearson_corr(rank_t, rank_v)
    if torch.isnan(rho):
        rho = torch.tensor(0.0, device=vals_t.device)
    alpha = a_min + (1.0 - rho.clamp(-1,1)) * 0.5 * (a_max - a_min)
    return float(alpha.item())

# ========== ANN 꼬리 재정렬 ==========
@torch.no_grad()
def _ann_tail_scores_and_cost(ncf, u, item_indices, device):

    if len(item_indices) == 0:
        return torch.empty(0, device=device), 0, 0
    uu = torch.full((len(item_indices),), u, device=device, dtype=torch.long)
    vv = torch.tensor(item_indices, device=device, dtype=torch.long)

    # forward (logit)
    g = ncf.get_gmf_only(uu, vv)             # [L, gdim]
    x = ncf.get_mlp_input(uu, vv)            # [L, 2*mlp_dim]
    macs = 0
    for layer in ncf.mlp_layers:
        if isinstance(layer, nn.Linear):
            macs += layer.in_features * layer.out_features * x.shape[0]
        x = layer(x)
    logits = ncf.out(torch.cat([g, x], dim=-1)).squeeze(-1)
    macs += (g.shape[1] + x.shape[1]) * 1 * x.shape[0]

    # emb mem access 근사
    mem_access = (2 * ncf.gmf_user_emb.embedding_dim + 2 * ncf.mlp_user_emb.embedding_dim)
    return logits.detach(), macs, mem_access

# ========== SNN 2-Stage Cascade ==========
@torch.no_grad()
def evaluate_snn_cascade(ncf, test_map, train_set, n_items, device,
                         T1=T1, T2=T2,
                         alpha=ALPHA_HYBRID_INIT,
                         s1_keep=(S1_KEEP_FIRST,S1_KEEP_HID,S1_KEEP_GMF,S1_KEEP_MLP),
                         s2_keep=(S2_KEEP_FIRST,S2_KEEP_HID,S2_KEEP_GMF,S2_KEEP_MLP),
                         dither_k=DITHER_K,
                         batch_size=BATCH_SIZE):

    ncf.eval()
    ncf = ncf.to(device)
    s_mlp_in = fit_scale_vec(ncf, train_set, device, p=TTFS_PERCENTILE, what="mlp_in")
    s_gmf    = fit_scale_vec(ncf, train_set, device, p=TTFS_PERCENTILE, what="gmf")

    # weights/bias
    linears=[m for m in ncf.mlp_layers if isinstance(m, nn.Linear)]
    W_list=[lin.weight.data.t().contiguous().to(device) for lin in linears] #  각 linear의 weight
    B_list=[lin.bias.data.contiguous().to(device) for lin in linears] # 각 linear의 bias

    w_full = ncf.out.weight.data.view(-1).to(device)
    w_g = w_full[:ncf.gmf_dim].contiguous()
    w_m = w_full[ncf.gmf_dim:].contiguous()
    b_out = (ncf.out.bias.data if ncf.out.bias is not None else torch.zeros(1)).to(device)



    alpha_s1, beta_g, beta_m = (alpha, 1.0, 1.0)
    users, fixed_negs = build_fixed_eval_pool(test_map, train_set, n_items, USERS_SAMPLE, NUM_NEGATIVES, SEED)

    hits, ndcgs=[],[]
    total_wall=0.0
    total_used=0
    total_possible=0
    total_mem=0.0
    total_energy=0.0

    # tail ANN cost 집계
    tail_macs_total = 0
    tail_mem_total  = 0

    use_amp=(device.type=='cuda')
    E_SYNAP = E_DISTANT_MEM_RW + E_LOCAL_MEM_RW + E_AC

    # 진단용
    dynM_sum=0; dynM_cnt=0; s1_recall_sum=0

    for u in tqdm(users, desc="SNN Cascade", leave=False):
        pos_items = test_map.get(u, [])
        if not pos_items: continue
        pos = pos_items[0]
        eval_items=[pos] + fixed_negs[u]
        B_all=len(eval_items)

        # --------- Stage-1 ---------
        t0=time.time()
        s1_scores = torch.full((B_all,), float('-inf'), device=device)

        for i in range(0, B_all, batch_size):
            sub = eval_items[i:i+batch_size]; B=len(sub)
            U=torch.tensor([u]*B, device=device, dtype=torch.long)
            I=torch.tensor(sub,   device=device, dtype=torch.long)

            with amp.autocast('cuda', enabled=use_amp):
                x0 = ncf.get_mlp_input(U,I)
                tp, mp, tn, mn = ttfs_times_posneg(x0, T1, s_mlp_in, GAMMA_MLP_IN_S1)
                t, used0, v0, L0 = layer_first_signed_inputs_amp(
                    tp, mp, tn, mn, W_list[0], B_list[0], T1,
                    neg_shift_frac=NEG_SHIFT_FRAC, keep_frac=s1_keep[0], dither_k=dither_k)
                u0 = int(used0.sum().item())
                total_used     += u0
                total_possible += used0.numel() * L0
                total_mem      += u0
                total_energy   += u0 * E_SYNAP

                for W,b in zip(W_list[1:], B_list[1:]):
                    t1, used1, v1, L1 = layer_relu_times_amp(
                        t, W, b, T1,
                        neg_shift_frac=NEG_SHIFT_FRAC, keep_frac=s1_keep[1], dither_k=dither_k)
                    u1 = int(used1.sum().item())
                    total_used     += u1; total_possible += used1.numel() * L1
                    total_mem      += u1; total_energy   += u1 * E_SYNAP
                    t = t1
                gvec = ncf.get_gmf_only(U,I)
                tg_pos, mg_pos, tg_neg, mg_neg = ttfs_times_posneg(gvec, T1, s_gmf, GAMMA_GMF_S1)
                tR, usedR, vTf, LR = readout_merge_signed_amp(
                    tg_pos, mg_pos, tg_neg, mg_neg, t, beta_m*w_m, beta_g*w_g, b_out, T1,
                    keep_frac_gmf=s1_keep[2], keep_frac_mlp=s1_keep[3], dither_k=dither_k)

                uR = int(usedR.sum().item())
                total_used     += uR; total_possible += usedR.numel() * LR
                total_mem      += uR; total_energy   += uR * E_SYNAP

                vt = vTf
                vt = vt / vt.abs().amax(dim=0, keepdim=False).clamp(min=1e-6)
                sc = alpha_s1 * (-(tR / float(T1))) + (1.0 - alpha_s1) * vt  # score계산
                s1_scores[i:i+B] = sc

        # 동적 Top-M 결정 (per-user)
        M_u = _pick_dynamic_topM(s1_scores, lo=TOP_M_LO, base=TOP_M_BASE, hi=TOP_M_HI,
                                 probe=MARGIN_PROBE, margin_lo=MARGIN_LO, margin_hi=MARGIN_HI)
        dynM_sum += M_u; dynM_cnt += 1

        # S1 Recall
        top_idx_u = torch.topk(s1_scores, k=min(M_u, B_all), largest=True).indices


        # --------- Stage-2  ---------
        s2_scores = torch.full((B_all,), float('-inf'), device=device)
        s2_vtonly = torch.full((B_all,), float('-inf'), device=device)  # vt-only 재정렬용

        # S2 adaptive alpha를 위한 저장 버퍼
        tR_all = torch.full((B_all,), float('nan'), device=device)
        vt_all = torch.full((B_all,), float('nan'), device=device)

        for j in range(0, top_idx_u.numel(), batch_size):
            idx_slice = top_idx_u[j:j+batch_size]
            sub = [eval_items[int(k)] for k in idx_slice.tolist()]
            B=len(sub)
            U=torch.tensor([u]*B, device=device, dtype=torch.long)
            I=torch.tensor(sub,   device=device, dtype=torch.long)

            with amp.autocast('cuda', enabled=use_amp):
                x0 = ncf.get_mlp_input(U,I)
                tp, mp, tn, mn = ttfs_times_posneg(x0, T2, s_mlp_in, GAMMA_MLP_IN_S2)
                t, used0, v0, L0 = layer_first_signed_inputs_amp(
                    tp, mp, tn, mn, W_list[0], B_list[0], T2,
                    neg_shift_frac=NEG_SHIFT_FRAC, keep_frac=s2_keep[0], dither_k=dither_k)
                u0 = int(used0.sum().item())
                total_used     += u0; total_possible += used0.numel() * L0
                total_mem      += u0; total_energy   += u0 * E_SYNAP

                for W,b in zip(W_list[1:], B_list[1:]):
                    t1, used1, v1, L1 = layer_relu_times_amp(
                        t, W, b, T2,
                        neg_shift_frac=NEG_SHIFT_FRAC, keep_frac=s2_keep[1], dither_k=dither_k)
                    u1 = int(used1.sum().item())
                    total_used     += u1; total_possible += used1.numel() * L1
                    total_mem      += u1; total_energy   += u1 * E_SYNAP
                    t = t1
                gvec = ncf.get_gmf_only(U,I)
                tg_pos, mg_pos, tg_neg, mg_neg = ttfs_times_posneg(gvec, T2, s_gmf, GAMMA_GMF_S2)
                tR, usedR, vTf, LR = readout_merge_signed_amp(
                    tg_pos, mg_pos, tg_neg, mg_neg, t, beta_m*w_m, beta_g*w_g, b_out, T2,
                    keep_frac_gmf=s2_keep[2], keep_frac_mlp=s2_keep[3], dither_k=dither_k)
                uR = int(usedR.sum().item())
                total_used     += uR; total_possible += usedR.numel() * LR
                total_mem      += uR; total_energy   += uR * E_SYNAP

                vt = vTf
                vt_norm = vt / vt.abs().amax(dim=0, keepdim=False).clamp(min=1e-6)

                # 임시 저장(Adaptive alpha 계산용)
                tR_all[idx_slice] = tR
                vt_all[idx_slice] = vt_norm

                # s2_scores는 사용자별 aplha 계산 후에 한 번에 채움
                s2_vtonly[idx_slice]  = vt_norm

        total_wall += time.time() - t0

        # 사용자별 Adaptive-alpha로 s2_scores 업데이트
        sel_idx = top_idx_u
        mask    = ~torch.isnan(tR_all[sel_idx])
        if mask.any():
            t_good = -(tR_all[sel_idx][mask] / float(T2)) #시간
            v_good =  (vt_all[sel_idx][mask]) #전위
            if ADAPTIVE_ALPHA:
                alpha_u = _adaptive_alpha_from_agreement(t_good, v_good)
            else:
                alpha_u = alpha
            s2_scores[sel_idx[mask]] = alpha_u * t_good + (1.0 - alpha_u) * v_good

        # 합산 점수: S2가 있는 곳은 S2, 나머지는 S1 그대로 사용
        final_scores = torch.where(s2_scores > float('-inf'), s2_scores, s1_scores)

        #  상위 L개 재정렬 (S2 vt가 있는 항목만)
        if TOP_L_VT_RERANK > 0:
            Lx = min(TOP_L_VT_RERANK, final_scores.numel())
            top_idx_final = torch.topk(final_scores, k=Lx, largest=True).indices
            have_vt = s2_vtonly[top_idx_final] > float('-inf')
            replace_idx = top_idx_final[have_vt]
            final_scores[replace_idx] = s2_vtonly[replace_idx]

        # 1차 최종 순위
        order = torch.argsort(final_scores, descending=True)


        # Top-L ANN 꼬리 재정렬 (상위 블록 내부 순서 교체)
        if ENABLE_ANN_TAIL and (order.numel() > 1):
            Ltail = min(TAIL_L, order.numel())
            tail_idx = order[:Ltail].detach().cpu().tolist()
            tail_items = [eval_items[int(k)] for k in tail_idx]

            # ANN 점수 & 비용
            t0_tail = time.time()
            ann_logits, macs_tail, mem_tail = _ann_tail_scores_and_cost(ncf, u, tail_items, device)
            total_wall += (time.time() - t0_tail)
            tail_macs_total += macs_tail
            tail_mem_total  += mem_tail

            # ANN 점수로 tail 내부만 재정렬
            ann_order = torch.argsort(ann_logits, descending=True).cpu().tolist()
            tail_idx_sorted = [tail_idx[i] for i in ann_order]


            # 최종 order의 상위 Ltail 부분만 교체
            order_final = order.clone()
            order_final[:Ltail] = torch.tensor(tail_idx_sorted, device=order.device, dtype=order.dtype)
        else:
            order_final = order

        ord_np = order_final.detach().cpu().numpy()
        if 0 in ord_np[:K]:
            hits.append(1.0); rk=np.where(ord_np[:K]==0)[0][0]; ndcgs.append(1.0/math.log2(rk+2))
        else:
            hits.append(0.0); ndcgs.append(0.0)

    denom=max(1,len(users))
    latency_ms_per_user = (total_wall/denom)*1000.0
    snn_spike_rate = float(total_used) / max(1.0, float(total_possible))

    # ANN tail 에너지
    tail_macs_user = float(tail_macs_total/denom)
    tail_mem_user  = float(tail_mem_total/denom)
    tail_energy_user = E_MAC * tail_macs_user + E_LOCAL_MEM_RW * tail_mem_user


    return {
        "Hit@K": float(np.mean(hits)) if hits else 0.0,
        "NDCG@K": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "Latency(ms/user)": float(latency_ms_per_user),
        "Memory Access Count (SNN full)": float(total_mem/denom),
        "Estimated Energy Score (SNN full+tail)": float((total_energy/denom) + tail_energy_user),
        "Ops (Synaptic) (SNN full)": float(total_used/denom),
        "SNN_spike_rate_eff (full)": float(snn_spike_rate),
        "Tail ANN MACs/user": tail_macs_user,
        "Tail ANN MemAccess/user": tail_mem_user,
        "Tail ANN Energy/user": tail_energy_user,
    }



# ========== Main ==========
@torch.no_grad()
def main(ncf_model, train_set, test_map, n_items, device):
    ann = evaluate_ann_full(ncf_model, test_map, train_set, n_items, device)
    snn = evaluate_snn_cascade(ncf_model, test_map, train_set, n_items, device,
                               T1=T1, T2=T2,
                               alpha=ALPHA_HYBRID_INIT,
                               s1_keep=(S1_KEEP_FIRST,S1_KEEP_HID,S1_KEEP_GMF,S1_KEEP_MLP),
                               s2_keep=(S2_KEEP_FIRST,S2_KEEP_HID,S2_KEEP_GMF,S2_KEEP_MLP),
                               dither_k=DITHER_K,
                               batch_size=BATCH_SIZE)


    print("\n\n" + "="*60)
    print("Full-SNN(TTFS, amp-preserving) 2-Stage Cascade vs ANN [ACTIVE denominator]")
    print("="*60)
    print(f"{'Metric':<40} | {'ANN':<14} | {'SNN (Cascade)':<14}")
    print("-"*60)
    print(f"{f'Hit@{K}':<40} | {ann['Hit@K']:<14.4f} | {snn['Hit@K']:<14.4f}")
    print(f"{f'NDCG@{K}':<40} | {ann['NDCG@K']:<14.4f} | {snn['NDCG@K']:<14.4f}")
    print("-"*60)
    print(f"{'Latency (ms/user)':<40} | {ann['Latency(ms/user)']:<14.2f} | {snn['Latency(ms/user)']:<14.2f}")
    print(f"{'Ops (MACs)/user':<40} | {ann['Ops (MACs)']:<14,.0f} | {'-':<14}")
    print(f"{'Memory Access/user (SNN only)':<40} | {'-':<14} | {snn['Memory Access Count (SNN full)']:<14,.0f}")
    print("-"*60)
    print(f"{'[theoretical] Energy/user (ANN full)':<40} | {ann['Estimated Energy Score (ANN full)']:<14,.0f} | {'-':<14}")
    print(f"{'[theoretical] Energy/user (SNN full)':<40} | {'-':<14} | {snn['Estimated Energy Score (SNN full+tail)']:<14,.0f}")
    print("-"*60)
    print(f"{'Ops (Synaptic)/user (SNN full)':<40} | {'-':<14} | {snn['Ops (Synaptic) (SNN full)']:<14,.0f}")
    print(f"{'SNN Spike Rate (full)':<40} | {'-':<14} | {snn['SNN_spike_rate_eff (full)']:<14.4f}")
    if ENABLE_ANN_TAIL:
        print("-"*60)
        print(f"{'Tail ANN MACs/user (extra)':<40} | {'-':<14} | {snn['Tail ANN MACs/user']:<14,.0f}")
        print(f"{'Tail ANN MemAccess/user (extra)':<40} | {'-':<14} | {snn['Tail ANN MemAccess/user']:<14,.0f}")
        print(f"{'Tail ANN Energy/user (extra)':<40} | {'-':<14} | {snn['Tail ANN Energy/user']:<14,.0f}")
    print("="*60)

# ========== Run ==========
if __name__ == "__main__":

  # multiline (gowalla)
  Rf_train, Rf_test, n_users, n_items = load_as_coo(DATA_ROOT, DATASET)
  train_set, test_map = build_trainset_testmap_from_coo(Rf_train, Rf_test)
  device = torch.device("cuda")

  ncf = load_neumf_checkpoint_safely(CKPT_IN, n_users, n_items, device).eval()
  main(ncf, train_set, test_map, n_items, device)


[embed] slice gmf_user_emb.weight 29858->29858
[embed] slice gmf_item_emb.weight 40981->40981
[embed] slice mlp_user_emb.weight 29858->29858
[embed] slice mlp_item_emb.weight 40981->40981
[CKPT] flexible-load + sanity-fix: users=29858, items=40981, gmf=8, mlp=32, mlp_layers=[64, 32, 16, 8]




Full-SNN(TTFS, amp-preserving) 2-Stage Cascade vs ANN [ACTIVE denominator]
Metric                                   | ANN            | SNN (Cascade) 
------------------------------------------------------------
Hit@10                                   | 0.4680         | 0.3680        
NDCG@10                                  | 0.2941         | 0.2092        
------------------------------------------------------------
Latency (ms/user)                        | 1.67           | 41.11         
Ops (MACs)/user                          | 6,806,800      | -             
Memory Access/user (SNN only)            | -              | 854,955       
------------------------------------------------------------
[theoretical] Energy/user (ANN full)     | 6,806,816      | -             
[theoretical] Energy/user (SNN full)     | -              | 1,077,243     
------------------------------------------------------------
Ops (Synaptic)/user (SNN full)           | -              | 854,955       
SNN 